In [1]:
!pip install conllu

In [2]:
import numpy as np
transition_prob_dict = np.load('transitionHMM.npy', allow_pickle='TRUE').item()
emission_prob_dict = np.load('emissionHMM.npy', allow_pickle='TRUE').item()


In [5]:
emission_prob_dict

{'las|DET': 0.062314279440825585,
 'reservas|NOUN': 0.0002456670474505902,
 'de|ADP': 0.4614486310043054,
 'oro|NOUN': 0.0004422006854110624,
 'y|CCONJ': 0.7767586602243879,
 'divisas|NOUN': 0.0001105501713527656,
 'rusia|PROPN': 0.0011091975831168453,
 'subieron|VERB': 0.0004360315994665025,
 '800|NUM': 0.000687096330905593,
 'millones|NOUN': 0.0062399430052449915,
 'dólares|NOUN': 0.002149586665192664,
 'el|DET': 0.3348459342750494,
 '26|NUM': 0.004259997251614677,
 'mayo|NOUN': 0.001154635123017774,
 'equivalían|VERB': 2.564891761567662e-05,
 'a|ADP': 0.15233418690379041,
 '19.100|NUM': 0.0001374192661811186,
 ',|PUNCT': 0.4600098135426889,
 'informó|VERB': 0.0020006155740227762,
 'hoy|ADV': 0.044629654523457456,
 'un|DET': 0.06363170606748152,
 'comunicado|NOUN': 0.0007247177899792411,
 'del|_': 0.5833744693454438,
 'banco|PROPN': 0.0023351528065617792,
 'central|PROPN': 0.0010216293528707784,
 '.|PUNCT': 0.26724918849550844,
 'según|ADP': 0.005571657708866815,
 'informe|NOUN': 0.0

In [8]:
# Identify every unique category 'upos' in the corpus
state_set = set([w.split('|')[1] for w in list(emission_prob_dict.keys()) if len(w.split('|')) == 2])
state_set

{'ADJ',
 'ADP',
 'ADV',
 'AUX',
 'CCONJ',
 'DET',
 'INTJ',
 'NOUN',
 'NUM',
 'PART',
 'PRON',
 'PROPN',
 'PUNCT',
 'SCONJ',
 'SYM',
 'VERB',
 'X',
 '_'}

In [9]:
# Enumerate each category to asign the Viterbi's matrix
tag_state_dict = {}
for i, state in enumerate(state_set):
    tag_state_dict[state] = i
tag_state_dict

{'VERB': 0,
 'AUX': 1,
 'SYM': 2,
 'PRON': 3,
 'PART': 4,
 'PROPN': 5,
 'NOUN': 6,
 'NUM': 7,
 'X': 8,
 'DET': 9,
 'SCONJ': 10,
 'INTJ': 11,
 'PUNCT': 12,
 'ADP': 13,
 '_': 14,
 'CCONJ': 15,
 'ADJ': 16,
 'ADV': 17}

In [10]:
# Initial Distribution
from conllu import parse_incr
word_list = []
data_file = open('UD_Spanish-AnCora/es_ancora-ud-train.conllu', 'r', encoding='utf-8')
count = 0 # Counter corpus length
init_tag_state_prob = {} # \rho_i^(o)
for tokenlist in parse_incr(data_file):
    count += 1
    tag = tokenlist[0]['upos']  # Get the first token's UPOS
    if tag in init_tag_state_prob.keys():
        init_tag_state_prob[tag] += 1
    else:
        init_tag_state_prob[tag] = 1

for key in init_tag_state_prob.keys():
    init_tag_state_prob[key] = init_tag_state_prob[key] / count  # Normalize the initial probabilities
init_tag_state_prob

{'DET': 0.3474487296143347,
 'ADP': 0.14887660110590048,
 'PROPN': 0.10450059494645482,
 'PRON': 0.0790228879400854,
 'CCONJ': 0.037096661300482954,
 'ADV': 0.0683138517533422,
 'PART': 0.002379785819276265,
 'VERB': 0.025827675509204173,
 'ADJ': 0.010149086582207601,
 'PUNCT': 0.09540141387275146,
 'NOUN': 0.024917757401833836,
 'NUM': 0.007349338559529643,
 '_': 0.009169174774270315,
 'SCONJ': 0.0271575558199762,
 'AUX': 0.0102890739833415,
 'INTJ': 0.0019598236158745713,
 'SYM': 0.00013998740113389796}

In [11]:
# check if probabilities sum to 1
sum(init_tag_state_prob.values())

1.0

# Construcción del algoritmo de Viterbi



Dada una secuencia de palabras $\{p_1, p_2, \dots, p_n \}$, y un conjunto de categorias gramaticales dadas por la convención `upos`, se considera la matriz de probabilidades de Viterbi así:

$$
\begin{array}{c c}
\begin{array}{c c c c}
\text{ADJ} \\
\text{ADV}\\
\text{PRON} \\
\vdots \\
{}
\end{array} 
&
\left[
\begin{array}{c c c c}
\nu_1(\text{ADJ}) & \nu_2(\text{ADJ}) & \dots  & \nu_n(\text{ADJ})\\
\nu_1(\text{ADV}) & \nu_2(\text{ADV}) & \dots  & \nu_n(\text{ADV})\\ 
\nu_1(\text{PRON}) & \nu_2(\text{PRON}) & \dots  & \nu_n(\text{PRON})\\
\vdots & \vdots & \dots & \vdots \\ \hdashline
p_1 & p_2 & \dots & p_n 
\end{array}
\right] 
\end{array}
$$

Donde las probabilidades de la primera columna (para una categoria $i$) están dadas por: 

$$
\nu_1(i) = \underbrace{\rho_i^{(0)}}_{\text{probabilidad inicial}} \times \underbrace{P(p_1 \vert i)}_{\text{emisión}}
$$

luego, para la segunda columna (dada una categoria $j$) serán: 

$$
\nu_2(j) = \max_i \{ \nu_1(i) \times \underbrace{P(j \vert i)}_{\text{transición}} \times \underbrace{P(p_2 \vert j)}_{\text{emisión}} \}
$$

así, en general las probabilidades para la columna $t$ estarán dadas por: 

$$
\nu_{t}(j) = \max_i \{ \overbrace{\nu_{t-1}(i)}^{\text{estado anterior}} \times \underbrace{P(j \vert i)}_{\text{transición}} \times \underbrace{P(p_t \vert j)}_{\text{emisión}} \}
$$

In [13]:
import nltk
nltk.download('punkt')
from nltk import word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     /home/brian02oriel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [16]:
def ViterbiMatrix(sequence, transition_prob_dict=transition_prob_dict, emission_prob_dict=emission_prob_dict, tag_state_dict=tag_state_dict, init_tag_state_prob=init_tag_state_prob):
    """
    Viterbi algorithm to find the most probable sequence of hidden states given an observed sequence.
    
    Args:
        sequence (list): List of observed tokens.
        transition_prob_dict (dict): Transition probabilities between states.
        emission_prob_dict (dict): Emission probabilities for each state.
        tag_state_dict (dict): Mapping of states to indices.
        init_tag_state_prob (dict): Initial state probabilities.
    
    Returns:
        list: Most probable sequence of hidden states.
    """
    seq = word_tokenize(sequence)
    viterbi_prob = np.zeros((len(tag_state_dict), len(seq)))  # Viterbi matrix
    
    # First column of the Viterbi matrix
    for key in tag_state_dict.keys():
        tag_row = tag_state_dict[key]
        word_tag = sequence[0].lower() + '|' + key
        if word_tag in emission_prob_dict.keys():
            viterbi_prob[tag_row][0] = init_tag_state_prob[key] * emission_prob_dict[word_tag]
    
    # Next columns of the Viterbi matrix
    for col in range(1, len(seq)):
        for key in tag_state_dict.keys():
            tag_row = tag_state_dict[key]
            word_tag = seq[col].lower() + '|' + key
            
            if word_tag in emission_prob_dict.keys():
                possible_probs = []
                for key2 in tag_state_dict.keys():
                    tag_row2 = tag_state_dict[key2]
                    tag_prevtag = key2 + '|' + key
                    if tag_prevtag in transition_prob_dict.keys():
                        if viterbi_prob[tag_row2][col - 1] > 0:
                        # Calculate the probability for the current state
                            possible_probs.append(viterbi_prob[tag_row2][col - 1] * transition_prob_dict[tag_prevtag] * emission_prob_dict[word_tag])
                if possible_probs:
                    viterbi_prob[tag_row][col] = max(possible_probs)
                else:
                    viterbi_prob[tag_row][col] = 0.0
    return viterbi_prob
matrix = ViterbiMatrix('el mundo es pequeño')
matrix


array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 6.23867632e-10, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.73022211e-08, 7.42017142e-13, 0.00000000e+00],
       [9.18220783e-07, 1.06059960e-07, 6.92072049e-13, 3.43196571e-16],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e

In [31]:
viterbiResultIndexes = []
for i in range(matrix.shape[1]):
    col_props = { j:  prob for j, prob in enumerate(matrix[:, i])if prob > 0}
    if col_props:
        max_key = max(col_props, key=col_props.get)
        viterbiResultIndexes.append(max_key)
    else:
        viterbiResultIndexes.append(None)

# Reverse tag_state_dict to map index to tag
index_tag_dict = {v: k for k, v in tag_state_dict.items()}
viterbiResult = [index_tag_dict[i] if i is not None else None for i in viterbiResultIndexes]
viterbiResult

['CCONJ', 'NOUN', 'AUX', 'ADJ']